In [ ]:
# New cell (index 0) - Compare two KDD datasets
import os
from pathlib import Path
import pandas as pd
import hashlib
import itertools

BASE = Path(r"C:\Users\Emery\Documents\GitHub\SEP740-CourseProject-G9-P19-AnomalyDetection\data\raw")
f1 = BASE / "kdd1999"
f2 = BASE / "kddcup.data_10_percent_corrected"

def file_stats(path):
    if not path.exists():
        return {"exists": False}
    size = path.stat().st_size
    # count lines efficiently
    with path.open("rb") as fh:
        lines = 0
        buf = fh.read(1024*1024)
        while buf:
            lines += buf.count(b"\n")
            buf = fh.read(1024*1024)
    # md5 of first 1MB for quick fingerprint
    with path.open("rb") as fh:
        sample = fh.read(1024*1024)
    md5 = hashlib.md5(sample).hexdigest()
    return {"exists": True, "size": size, "lines": lines, "md5_sample": md5}

def read_table(path, nrows=None):
    if not path.exists():
        raise FileNotFoundError(path)
    try:
        return pd.read_csv(path, header=None, low_memory=False, nrows=nrows)
    except Exception:
        # fallback: try whitespace delim
        return pd.read_csv(path, header=None, delim_whitespace=True, low_memory=False, nrows=nrows)

def summarize_df(df, name, top_k=5):
    s = {}
    s["name"] = name
    s["shape"] = df.shape
    s["dtypes"] = df.dtypes.value_counts().to_dict()
    s["missing_per_col"] = df.isna().sum().head(top_k).to_dict()
    s["unique_counts_head"] = {i: int(df[i].nunique(dropna=False)) for i in range(min(5, df.shape[1]))}
    # label/last column distribution
    last = df.columns[-1]
    s["label_counts_top"] = df[last].value_counts().head(top_k).to_dict()
    return s

def row_hashes(df):
    # deterministic hash per row
    return df.astype(str).agg("|".join, axis=1).apply(lambda x: hashlib.md5(x.encode("utf-8")).hexdigest())

# gather file-level stats
stats1 = file_stats(f1)
stats2 = file_stats(f2)

print("FILE STATS")
print(f"{f1}: {stats1}")
print(f"{f2}: {stats2}\n")

# load sample / full depending on size
def smart_read(path):
    if not path.exists():
        return None
    if path.stat().st_size > 200_000_000:  # >200MB: sample first 100k rows
        return read_table(path, nrows=100_000)
    return read_table(path)

df1 = smart_read(f1)
df2 = smart_read(f2)

if df1 is None or df2 is None:
    raise SystemExit("One or both files not found.")

sum1 = summarize_df(df1, "kdd1999")
sum2 = summarize_df(df2, "kddcup.data_10_percent_corrected")

print("DATAFRAME SUMMARIES")
print(sum1)
print(sum2)
print()

# Compare column counts and shapes
print("COMPARISONS")
print("shapes:", sum1["shape"], sum2["shape"])
print("num_columns:", sum1["shape"][1], sum2["shape"][1])
if sum1["shape"][1] != sum2["shape"][1]:
    print("-> Column count differs.")

# Compare label sets (last column)
lab1 = set(df1[df1.columns[-1]].unique())
lab2 = set(df2[df2.columns[-1]].unique())
print("labels in file1 not in file2:", sorted(list(lab1 - lab2))[:20])
print("labels in file2 not in file1:", sorted(list(lab2 - lab1))[:20])

# Row-level overlap on sample to avoid memory blowup
def sample_row_hash_set(path, sample_n=200000, seed=0):
    df = read_table(path, nrows=sample_n)
    return set(row_hashes(df).tolist())

h1 = sample_row_hash_set(f1, sample_n=50000)
h2 = sample_row_hash_set(f2, sample_n=50000)
intersection = len(h1 & h2)
print(f"approx row-hash sample intersection (first 50k rows each): {intersection}")

# Column-level unique counts comparison (first 10 columns)
cols = min(df1.shape[1], df2.shape[1], 10)
for i in range(cols):
    u1 = df1[i].nunique(dropna=False)
    u2 = df2[i].nunique(dropna=False)
    if u1 != u2:
        print(f"col {i}: unique counts differ -> file1: {u1}, file2: {u2}")

print("\nDone.")